# Settimana 14 — Deployment nel Cloud

**Obiettivo:** portare la DocBot API dalla tua macchina a un URL pubblico su Railway o Render.

Questo notebook è diverso dai precedenti: invece di eseguire codice Python interattivo,
genera i file di cui hai bisogno e ti guida passo dopo passo nel processo di deployment.

**Alla fine avrai:**
- Un repository GitHub pronto per CI/CD
- Una pipeline GitHub Actions che esegue i test prima di distribuire
- La DocBot API distribuita su Railway o Render con URL pubblico
- Uno script di smoke test che verifica il deployment

**Prerequisiti:**
- `proyectos/docbot-api/` completo (settimana 13)
- Account su [Railway](https://railway.app) o [Render](https://render.com) (entrambi gratuiti)
- Account GitHub
- Railway CLI installato: `npm install -g @railway/cli` (opzionale, per il deploy da terminale)

## Cella 1 — Preparare il repository

Prima verifica che il progetto sia in ordine prima di caricarlo su GitHub.

In [ ]:
import os
import subprocess
from pathlib import Path

# Percorso del progetto docbot-api
PROJECT_DIR = Path("../proyectos/docbot-api").resolve()
print(f"Progetto: {PROJECT_DIR}")
print(f"Esiste: {PROJECT_DIR.exists()}")

# File che devono esistere
ARCHIVOS_REQUERIDOS = [
    "api.py",
    "models.py",
    "Dockerfile",
    ".dockerignore",
    "requirements.txt",
    "test_api.py",
]

print("\nChecklist dei file:")
todos_ok = True
for archivo in ARCHIVOS_REQUERIDOS:
    ruta = PROJECT_DIR / archivo
    estado = "✓" if ruta.exists() else "✗ MANCANTE"
    print(f"  {estado}  {archivo}")
    if not ruta.exists():
        todos_ok = False

if todos_ok:
    print("\n✓ Progetto pronto per il deployment")
else:
    print("\n✗ Mancano dei file — esegui prima il notebook della settimana 13")

## Cella 2 — Generare .gitignore

Il `.gitignore` impedisce che segreti e dati locali finiscano su GitHub.

In [ ]:
GITIGNORE = """# Segreti
.env
*.env

# Dati persistenti (vanno nei volumi cloud, non nel repo)
chroma_db/
sesiones/
evaluaciones/

# Python
__pycache__/
*.pyc
*.pyo
.venv/
venv/
*.egg-info/
dist/
build/

# Jupyter
.ipynb_checkpoints/

# IDE
.vscode/
.idea/
*.swp

# macOS
.DS_Store
"""

ruta_gitignore = PROJECT_DIR / ".gitignore"
ruta_gitignore.write_text(GITIGNORE)
print(f"✓ .gitignore generato in {ruta_gitignore}")

# Verifica che .env non sia nel progetto (se esiste, non deve essere caricato)
env_file = PROJECT_DIR / ".env"
if env_file.exists():
    print("⚠️  Esiste un .env — è nel .gitignore, non verrà caricato su GitHub")
else:
    print("✓ Non c'è nessun .env nel progetto (corretto)")

## Cella 3 — Pipeline di GitHub Actions

Questa pipeline viene eseguita a ogni `git push` su `main`:
1. Esegue i test
2. Se passano, distribuisce su Railway

**Nota:** lo step di Railway richiede un token. Nel passaggio di configurazione di Railway vedrai come ottenerlo.

In [ ]:
import os

WORKFLOW_DIR = PROJECT_DIR / ".github" / "workflows"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)

DEPLOY_YML = """name: Test and Deploy

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run tests
        run: pytest test_api.py -v --tb=short
        env:
          # I test di integrazione con l'LLM richiedono la key reale.
          # Definisci OPENAI_API_KEY in GitHub → Settings → Secrets.
          # Se i tuoi test mockano l'LLM, qui puoi usare un valore fittizio.
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}

  deploy:
    needs: test
    runs-on: ubuntu-latest
    # Distribuisce solo su push a main, non nelle PR
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    steps:
      - uses: actions/checkout@v4

      - name: Install Railway CLI
        run: npm install -g @railway/cli

      - name: Deploy to Railway
        run: railway up --detach
        env:
          RAILWAY_TOKEN: ${{ secrets.RAILWAY_TOKEN }}
"""

ruta_workflow = WORKFLOW_DIR / "deploy.yml"
ruta_workflow.write_text(DEPLOY_YML)
print(f"✓ Pipeline generata: {ruta_workflow}")
print()
print("Prossimi passi:")
print("  1. Crea il repository su GitHub")
print("  2. Aggiungi OPENAI_API_KEY e RAILWAY_TOKEN in GitHub → Settings → Secrets")
print("  3. git push origin main → la pipeline si attiva automaticamente")

## Cella 4 — Healthcheck migliorato nel Dockerfile

Il Dockerfile della settimana 13 espone la porta ma non ha HEALTHCHECK.
Lo aggiungiamo affinché Railway/Render sappia quando il container è pronto.

In [ ]:
DOCKERFILE = """FROM python:3.11-slim

WORKDIR /app

# Installa le dipendenze per prime — vengono messe in cache se requirements.txt non cambia
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copia il codice
COPY . .

# Crea le directory per i volumi
RUN mkdir -p chroma_db sesiones evaluaciones

EXPOSE 8000

# Railway/Render interroga questo endpoint per sapere se il container è vivo.
# --interval=30s: ogni 30 secondi
# --timeout=10s:  se non risponde entro 10s, conta come fallimento
# --start-period=15s: attende 15s prima del primo check (tempo di avvio)
# --retries=3: 3 fallimenti consecutivi → il container viene marcato come unhealthy
HEALTHCHECK --interval=30s --timeout=10s --start-period=15s --retries=3 \\
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]
"""

ruta_dockerfile = PROJECT_DIR / "Dockerfile"
ruta_dockerfile.write_text(DOCKERFILE)
print(f"✓ Dockerfile aggiornato con HEALTHCHECK: {ruta_dockerfile}")

# Mostra il Dockerfile
print()
print(ruta_dockerfile.read_text())

## Cella 5 — Script di smoke test

Uno smoke test verifica che l'API distribuita risponda correttamente.
Lo esegui dopo ogni deployment con l'URL reale di produzione.

In [ ]:
SMOKE_TEST = '''#!/usr/bin/env python3
"""
Smoke test per la DocBot API in produzione.
Uso: python smoke_test.py https://tua-api.railway.app
"""
import sys
import time
import json
import urllib.request
import urllib.error

def get(url: str, timeout: int = 10) -> dict:
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return json.loads(r.read())

def post(url: str, data: dict, timeout: int = 30) -> dict:
    body = json.dumps(data).encode()
    req = urllib.request.Request(
        url, data=body,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def test(nombre: str, fn):
    try:
        inicio = time.time()
        resultado = fn()
        latencia = (time.time() - inicio) * 1000
        print(f"  ✓  {nombre} ({latencia:.0f}ms)")
        return resultado
    except Exception as e:
        print(f"  ✗  {nombre}: {e}")
        return None

def main():
    if len(sys.argv) < 2:
        print("Uso: python smoke_test.py https://tua-api.railway.app")
        sys.exit(1)

    base = sys.argv[1].rstrip("/")
    print(f"\nSmoke test → {base}\n")

    fallos = 0

    # 1. Health check
    r = test("GET /health", lambda: get(f"{base}/health"))
    if r is None or r.get("status") != "ok":
        print("    → L'API non risponde o status != ok")
        fallos += 1

    # 2. Ingestione di un articolo di esempio
    r = test("POST /ingest (Wikipedia: FastAPI)", lambda: post(
        f"{base}/ingest",
        {"fuentes": [{"tipo": "wikipedia", "valor": "FastAPI",
                       "titulo": "FastAPI", "categoria": "test"}]}
    ))
    if r is None or not r.get("ok"):
        print("    → Fallimento nell'ingestione")
        fallos += 1

    # 3. Domanda sul corpus
    r = test("POST /chat (domanda sul corpus)", lambda: post(
        f"{base}/chat",
        {"pregunta": "What is FastAPI?", "user_id": "smoke_test"}
    ))
    if r is None or not r.get("respuesta"):
        print("    → Fallimento nella chat")
        fallos += 1
    elif r:
        print(f"    → respondida={r.get('respondida')} latenza={r.get('latencia_ms', 0):.0f}ms")

    # 4. Validazione: request non valido deve dare 422
    try:
        post(f"{base}/chat", {"pregunta": "", "user_id": "smoke_test"})
        print("  ✗  POST /chat (domanda vuota) → dovrebbe dare 422 ma non l'ha fatto")
        fallos += 1
    except urllib.error.HTTPError as e:
        if e.code == 422:
            print("  ✓  POST /chat (domanda vuota) → 422 corretto")
        else:
            print(f"  ✗  POST /chat (domanda vuota) → codice inatteso {e.code}")
            fallos += 1

    # Risultato
    print()
    if fallos == 0:
        print("✓ Tutti i test sono passati — l'API è operativa")
    else:
        print(f"✗ {fallos} test falliti")
        sys.exit(1)

if __name__ == "__main__":
    main()
'''

ruta_smoke = PROJECT_DIR / "smoke_test.py"
ruta_smoke.write_text(SMOKE_TEST)
print(f"✓ smoke_test.py generato in {ruta_smoke}")
print()
print("Uso dopo il deployment:")
print("  python smoke_test.py https://tua-api.railway.app")

## Cella 6 — Guida passo dopo passo: deployment su Railway

Esegui questa cella per vedere le istruzioni complete di deployment.

In [ ]:
GUIA = """
============================================================
 GUIDA AL DEPLOYMENT — Railway
============================================================

PASSO 1 — Creare il repository su GitHub
---------------------------------------
  cd proyectos/docbot-api
  git init
  git add .
  git commit -m "DocBot API v1.0"
  # Crea il repo su github.com/new, poi:
  git remote add origin https://github.com/TUO_UTENTE/docbot-api.git
  git push -u origin main

PASSO 2 — Creare il progetto su Railway
--------------------------------------
  1. Vai su railway.app e fai login con GitHub
  2. New Project → Deploy from GitHub repo → seleziona docbot-api
  3. Railway rileva automaticamente il Dockerfile

PASSO 3 — Configurare le variabili d'ambiente
-----------------------------------------
  Su Railway → il tuo servizio → Variables → Add Variable:
    OPENAI_API_KEY = sk-...
    API_KEYS       = chiave1,chiave2   (opzionale, se hai attivato l'auth)
    LOG_LEVEL      = INFO

PASSO 4 — Ottenere il token di Railway per GitHub Actions
---------------------------------------------------------
  railway.app → Account Settings → Tokens → New Token
  Nome: github-actions
  Copia il token.

  Su GitHub → il tuo repo → Settings → Secrets → New secret:
    RAILWAY_TOKEN  = (incolla il token)
    OPENAI_API_KEY = sk-...

PASSO 5 — Primo deployment
---------------------------
  Railway sta già costruendo l'immagine dal push iniziale.
  Attendi ~2 minuti e Railway ti fornisce un URL pubblico:
    https://docbot-api-production.railway.app

PASSO 6 — Verificare con lo smoke test
-------------------------------------
  python smoke_test.py https://docbot-api-production.railway.app

PASSO 7 — CI/CD in azione
-------------------------
  Modifica un file qualsiasi e fai git push.
  GitHub Actions → esegue i test → se passano → Railway distribuisce.
  Puoi vedere la pipeline in GitHub → Actions tab.

============================================================
 ALTERNATIVA: Render (free tier)
============================================================

  1. render.com → New → Web Service → Connect GitHub repo
  2. Environment: Docker
  3. Aggiungi le variabili d'ambiente nel pannello
  4. Deploy → attendi ~3 minuti

  ⚠️  Il free tier di Render mette l'istanza in stop dopo 15 min di inattività.
  Il primo request dopo lo sleep richiede ~30s.
  Per le demo, usa Railway (non va in stop con il piano Hobby a $5/mese).
"""

print(GUIA)

## Cella 7 — Verificare il deployment in produzione

Una volta distribuita l'API, esegui questa cella con il tuo URL reale per verificare che tutto funzioni.

In [ ]:
import urllib.request
import urllib.error
import json
import time

# TODO: sostituisci con il tuo URL reale di Railway o Render
API_URL = "https://TUA-API.railway.app"

# Se non hai ancora l'URL, questa cella mostra un avviso
if "TUA-API" in API_URL:
    print("⚠️  Sostituisci API_URL con l'URL reale del tuo deployment")
    print("    Esempio: API_URL = 'https://docbot-api-production.railway.app'")
else:
    print(f"Verifica di {API_URL}...\n")

    try:
        inicio = time.time()
        with urllib.request.urlopen(f"{API_URL}/health", timeout=15) as r:
            data = json.loads(r.read())
            latencia_ms = (time.time() - inicio) * 1000

        print(f"✓ /health → {data}")
        print(f"  Latenza: {latencia_ms:.0f}ms")

        if latencia_ms > 2000:
            print("  ⚠️  Latenza alta — probabilmente l'istanza era in stop (Render free tier)")
        else:
            print("  → L'API risponde velocemente, istanza attiva")

    except urllib.error.URLError as e:
        print(f"✗ Impossibile connettersi: {e}")
        print("  Controlla che l'URL sia corretto e il servizio sia distribuito")

## Cella 8 — Monitor di costi e latenza

Un monitor di base che chiama l'endpoint `/metrics/{user_id}` e mostra un riepilogo.
Utile per tenere traccia del costo durante le demo ai clienti.

In [ ]:
import urllib.request
import json

# TODO: sostituisci con il tuo URL e user_id reale
API_URL  = "https://TUA-API.railway.app"
USER_ID  = "test_user"

if "TUA-API" in API_URL:
    # Simulazione con dati di esempio per vedere il formato
    print("[Modalità simulazione — sostituisci API_URL con l'URL reale]\n")
    metricas = {
        "user_id": "test_user",
        "n_preguntas": 12,
        "n_respondidas": 10,
        "latencia_media_ms": 1843.2,
        "faithfulness_media": 0.87,
    }
else:
    with urllib.request.urlopen(f"{API_URL}/metrics/{USER_ID}", timeout=10) as r:
        metricas = json.loads(r.read())

# Mostra il riepilogo
print("=" * 40)
print(f" Metriche — utente: {metricas.get('user_id', USER_ID)}")
print("=" * 40)

n_p = metricas.get("n_preguntas", 0)
n_r = metricas.get("n_respondidas", 0)
tasa = (n_r / n_p * 100) if n_p > 0 else 0

print(f"  Domande totali:      {n_p}")
print(f"  Risposte date:       {n_r} ({tasa:.0f}%)")
print(f"  Latenza media:       {metricas.get('latencia_media_ms', 0):.0f}ms")
print(f"  Faithfulness media:  {metricas.get('faithfulness_media', 0):.2f}")

# Stima del costo con gpt-4o-mini
# ~500 token per domanda (prompt + risposta) a $0.15/1M input + $0.60/1M output
coste_estimado = n_p * 500 * (0.15 + 0.60) / 2 / 1_000_000
print(f"  Costo stimato:       ${coste_estimado:.4f}")
print()

# Alert
if tasa < 70:
    print("⚠️  Tasso di risposta basso (<70%) — considera di ampliare il corpus")
if metricas.get("latencia_media_ms", 0) > 3000:
    print("⚠️  Latenza alta (>3s) — controlla i log su Railway/Render")
if metricas.get("faithfulness_media", 1) < 0.7:
    print("⚠️  Faithfulness bassa (<0.7) — controlla il prompt e i chunk recuperati")

## Sfida — Aggiungere un endpoint `/status` con info sul deployment

L'health check attuale dice solo `{"status": "ok"}`. Per la produzione è utile avere più informazioni:
versione, tempo di attività, numero di utenti attivi.

**Implementa** l'endpoint `GET /status` in `api.py` che restituisca:

```json
{
  "status": "ok",
  "version": "1.0.0",
  "uptime_seconds": 3842,
  "usuarios_activos": 3,
  "modelo": "gpt-4o-mini"
}
```

Suggerimenti:
- Salva `_start_time = time.time()` nel lifespan all'avvio
- `uptime_seconds = int(time.time() - _start_time)`
- `usuarios_activos = len(_bots)` (numero di bot nel pool)
- Aggiungi il modello come costante in `api.py`

Una volta implementato, verifica che lo smoke test lo rilevi aggiungendo un caso:
```python
r = test("GET /status", lambda: get(f"{base}/status"))
assert r["uptime_seconds"] >= 0
```

In [ ]:
# Scheletro per la sfida
# Implementa l'endpoint /status in api.py

SKELETON_STATUS = '''
# In api.py — aggiungi questo:

import time

MODELO   = "gpt-4o-mini"
VERSION  = "1.0.0"
_start_time: float = 0.0

@asynccontextmanager
async def lifespan(app: FastAPI):
    global _start_time
    _start_time = time.time()   # <-- aggiungi questa riga al lifespan esistente
    # ... resto del lifespan ...
    yield
    # ... shutdown ...


class StatusResponse(BaseModel):
    status:           str
    version:          str
    uptime_seconds:   int
    usuarios_activos: int
    modelo:           str


@app.get("/status", response_model=StatusResponse)
async def status() -> StatusResponse:
    # TODO: implementa il corpo dell'endpoint
    pass
'''

print(SKELETON_STATUS)

## Riepilogo della settimana 14

| Concetto | Dove lo hai applicato |
|---------|-------------------|
| `.gitignore` con segreti e dati | Cella 2 |
| Pipeline CI/CD con GitHub Actions | Cella 3 |
| HEALTHCHECK nel Dockerfile | Cella 4 |
| Smoke test per la produzione | Cella 5 |
| Guida al deployment Railway/Render | Cella 6 |
| Verifica del deployment in produzione | Cella 7 |
| Monitor di costi e latenza | Cella 8 |
| Endpoint `/status` esteso | Sfida |

**Blocco 6 completato.** DocBot ha ora un URL pubblico, test automatici e deployment continuo.

La settimana 15 apre il Blocco 7: **Progetto 2 — Agente autonomo** per un caso d'uso reale di business.